In [1]:
import logging

logging.getLogger('fastf1').setLevel(logging.ERROR)

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
import fastf1
import pandas as pd
import numpy as np
from pathlib import Path
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

fastf1.Cache.enable_cache('cache')

In [3]:
df_drivers = pd.read_csv('data/Model2/drivers.csv').convert_dtypes()
df_laps = pd.read_csv('data/Model2/laps.csv').convert_dtypes()
df_track_status = pd.read_csv('data/Model2/track_status.csv').convert_dtypes()
df_weather = pd.read_csv('data/Model2/weather.csv').convert_dtypes()

C:\Users\Ogi\AppData\Local\Temp\ipykernel_14020\2993207287.py:2: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df_laps = pd.read_csv('data/Model2/laps.csv').convert_dtypes()


In [4]:
for col in df_laps.select_dtypes(include=['string','boolean']).columns:
    if df_laps[col].dtype.name == 'boolean':
        df_laps[col] = df_laps[col].astype('float')
    else:
        df_laps[col] = df_laps[col].astype('object')

In [5]:
df_laps = df_laps.replace({pd.NA: np.nan})

In [6]:
df_laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Season,Round,EventName
0,0 days 00:34:06.302000,GAS,10,0 days 00:01:19.106000,1,1,NaN,NaN,NaN,0 days 00:00:33.352000,0 days 00:00:23.247000,NaN,0 days 00:33:43.079000,0 days 00:34:06.511000,309,217,274,290,0.0,MEDIUM,1,1.0,AlphaTauri,0 days 00:32:47.006000,<NA>,1,12,<NA>,<NA>,0.0,0.0,2020,0,Pre-Season Test 1
1,0 days 00:35:18.714000,GAS,10,0 days 00:01:12.412000,2,1,NaN,NaN,0 days 00:00:17.960000,0 days 00:00:32.228000,0 days 00:00:22.224000,0 days 00:34:24.262000,0 days 00:34:56.490000,0 days 00:35:18.714000,282,226,270,292,1.0,MEDIUM,2,1.0,AlphaTauri,0 days 00:34:06.302000,<NA>,1,12,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1
2,0 days 00:36:30.025000,GAS,10,0 days 00:01:11.311000,3,1,NaN,NaN,0 days 00:00:17.513000,0 days 00:00:31.717000,0 days 00:00:22.081000,0 days 00:35:36.227000,0 days 00:36:07.944000,0 days 00:36:30.025000,307,224,277,311,1.0,MEDIUM,3,1.0,AlphaTauri,0 days 00:35:18.714000,<NA>,1,12,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1
3,0 days 00:37:40.750000,GAS,10,0 days 00:01:10.725000,4,1,NaN,NaN,0 days 00:00:17.410000,0 days 00:00:31.510000,0 days 00:00:21.805000,0 days 00:36:47.435000,0 days 00:37:18.945000,0 days 00:37:40.750000,309,218,275,309,1.0,MEDIUM,4,1.0,AlphaTauri,0 days 00:36:30.025000,<NA>,1,12,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1
4,0 days 00:38:52.261000,GAS,10,0 days 00:01:11.511000,5,1,NaN,NaN,0 days 00:00:17.464000,0 days 00:00:31.827000,0 days 00:00:22.220000,0 days 00:37:58.214000,0 days 00:38:30.041000,0 days 00:38:52.261000,303,219,268,296,0.0,MEDIUM,5,1.0,AlphaTauri,0 days 00:37:40.750000,<NA>,1,12,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1


In [7]:
df_drivers.head()

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps,Season,Round
0,77,V BOTTAS,BOT,bottas,Mercedes,00D2BE,mercedes,Valtteri,Bottas,Valtteri Bottas,https://www.formula1.com/content/dam/fom-websi...,<NA>,1,1,1,<NA>,<NA>,<NA>,0 days 01:30:55.739000,Finished,25.0,71,2020,0
1,16,C LECLERC,LEC,leclerc,Ferrari,DC0000,ferrari,Charles,Leclerc,Charles Leclerc,https://www.formula1.com/content/dam/fom-websi...,<NA>,2,2,7,<NA>,<NA>,<NA>,0 days 00:00:02.700000,Finished,18.0,71,2020,0
2,4,L NORRIS,NOR,norris,McLaren,FF8700,mclaren,Lando,Norris,Lando Norris,https://www.formula1.com/content/dam/fom-websi...,<NA>,3,3,3,<NA>,<NA>,<NA>,0 days 00:00:05.491000,Finished,16.0,71,2020,0
3,44,L HAMILTON,HAM,hamilton,Mercedes,00D2BE,mercedes,Lewis,Hamilton,Lewis Hamilton,https://www.formula1.com/content/dam/fom-websi...,<NA>,4,4,5,<NA>,<NA>,<NA>,0 days 00:00:05.689000,Finished,12.0,71,2020,0
4,55,C SAINZ,SAI,sainz,McLaren,FF8700,mclaren,Carlos,Sainz,Carlos Sainz,https://www.formula1.com/content/dam/fom-websi...,<NA>,5,5,8,<NA>,<NA>,<NA>,0 days 00:00:08.903000,Finished,10.0,71,2020,0


In [8]:
df_track_status.head()

,Time,Status,Message,Season,Round
0,0 days 00:07:56.420000,1,AllClear,2020,0
1,0 days 00:56:21.333000,2,Yellow,2020,0
2,0 days 00:56:30.327000,1,AllClear,2020,0
3,0 days 01:01:52.836000,2,Yellow,2020,0
4,0 days 01:02:31.497000,4,SCDeployed,2020,0


In [9]:
df_weather.head()

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,Season,Round
0,0 days 00:00:38.669000,27.5,35.1,939.4,False,55.6,99,1.7,2020,0
1,0 days 00:01:38.684000,27.1,36.5,939.4,False,55.1,106,1.9,2020,0
2,0 days 00:02:38.655000,27.0,36.2,939.3,False,54.2,111,2.1,2020,0
3,0 days 00:03:38.669000,26.9,36.1,939.3,False,54.2,165,1.0,2020,0
4,0 days 00:04:38.678000,27.2,35.8,939.3,False,54.2,83,1.2,2020,0
